# Phase 2: Imbalanced Data & Feature Engineering (클래스 불균형 해소 및 피처 추출)

이 단계에서는 전처리된 CMP 공정 데이터를 분석 알고리즘이 잘 학습할 수 있도록 유용한 **피처(Feature)**들을 발굴하고, 실제 제조 현장 데이터의 가장 큰 장애물인 **극심한 클래스 불균형(Class Imbalance)** 문제를 해소하기 위한 오버샘플링 기술을 배웁니다.

## 🎯 실습 목표
1. **시계열 도메인 피처 엔지니어링**: Rolling 통계량(평균, 편차) 및 가공 상태 결합 피처 생성
2. **FFT (Fast Fourier Transform)**: 고주파 마찰 진동 센서의 주파수 대역 에너지 변환 분석
3. **인위적 고장 라벨 시뮬레이션**: 특정 가혹 물리 상태 결합 조건에 따른 고장 시나리오 라벨 생성
4. **SMOTE 기법 적용**: 소수 고장 샘플을 선형 보간 기반으로 복제하여 불균형 문제 극복

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler

# 시각화 설정
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 이전 단계 저장 파일 로드
try:
    df_phase1 = pd.read_csv('./data/cmp_phase1_cleaned.csv')
    print("Phase 1 전처리 데이터 로드 성공!")
except FileNotFoundError:
    print("Phase 1 파일이 없습니다. 01_timeseries_preprocessing.ipynb를 먼저 실행해 주세요.")

## 1. 시계열 피처 추출 (Feature Engineering)
센서의 절대 수치값 외에, 일정 윈도우 크기 동안의 국소적 변동성(Rolling Std)과 장비 과열 및 기계적 부하를 나타내는 파생 피처들을 설계합니다.

In [ ]:
df_features = df_phase1.copy()

# 1. 진동 변동성 지표 : vibration_kalman의 5초 단위 rolling standard deviation
df_features['vibration_std_5s'] = df_features['vibration_kalman'].rolling(window=5, min_periods=1).std().fillna(0)

# 2. 모터 전류 변동성 지표 
df_features['current_std_5s'] = df_features['motor_current'].rolling(window=5, min_periods=1).std().fillna(0)

# 3. 물리 도메인 유도 파생 피처 (Mechanical Power proxy) : 압력 * RPM / 전류
# 장비 가공 능률 지표로 활용
df_features['mechanical_load'] = (df_features['pressure'] * df_features['spindle_speed']) / (df_features['motor_current'] + 1e-5)

print(df_features[['vibration_kalman', 'vibration_std_5s', 'mechanical_load']].head())

## 2. 주파수 영역 변환 (FFT: Fast Fourier Transform)
마찰이나 설비 충격으로 발생하는 고주파 진동 신호는 주파수 영역에서 분석할 때 설비 손상 주파수를 극적으로 탐지할 수 있습니다.

In [ ]:
# 가상의 고주파 진동 신호 500개 샘플을 추출하여 주파수 도메인 에너지 플롯 구현
signal = df_phase1['vibration_kalman'].values[:500]
N = len(signal)
T = 0.1 # 샘플링 주기 (10Hz 데이터 기준 1초에 10개이므로 0.1초)

yf = fft(signal)
xf = fftfreq(N, T)[:N//2]
spectral_density = 2.0/N * np.abs(yf[0:N//2])

plt.figure(figsize=(12, 4))
plt.plot(xf, spectral_density, color='crimson')
plt.title('CMP Vibration Frequency Spectrum (FFT)', fontsize=13)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Spectral Energy')
plt.show()

## 3. 반도체 CMP 공정 고장 시나리오 라벨링
현업 데이터셋에는 고장 사례가 매우 적기 때문에, 특정 물리적 부하량(압력, 전류, 진동)이 비정상적으로 치솟았을 때 기계적 스트레스로 인한 고장이 발생한다고 정의하여 **불균형 고장 라벨**을 생성합니다.

In [ ]:
# 압력이 크고 진동 변동성이 크며 전류값이 과도하게 큰 조합 조건 ➔ 고장(1), 정상(0)
# 정상 샘플 중 약 2~3% 미만만 고장이 발생하도록 임계 조건 설계
load_threshold = df_features['mechanical_load'].quantile(0.97)
vibration_threshold = df_features['vibration_std_5s'].quantile(0.95)

df_features['failure'] = 0
failure_cond = (df_features['mechanical_load'] > load_threshold) & (df_features['vibration_std_5s'] > vibration_threshold)
df_features.loc[failure_cond, 'failure'] = 1

class_counts = df_features['failure'].value_counts()
class_ratio = df_features['failure'].value_counts(normalize=True) * 100

print("--- 클래스 분포 ---")
for c in class_counts.index:
    print(f"Class {c}: {class_counts[c]}건 ({class_ratio[c]:.2f}%)")

## 4. SMOTE (Synthetic Minority Over-sampling Technique) 오버샘플링
학습용 독립변수(`X`)와 타겟변수(`y`)를 설정하고, `imbalanced-learn` 라이브러리의 SMOTE를 사용해 소수 고장 클래스를 동등한 수준으로 끌어올립니다.

In [ ]:
from sklearn.model_selection import train_test_split

features_list = ['spindle_speed', 'pressure', 'slurry_flow', 'motor_current', 
                 'vibration_kalman', 'vibration_std_5s', 'current_std_5s', 'mechanical_load']

X = df_features[features_list]
y = df_features['failure']

# 학습용과 검증용으로 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    print("--- SMOTE 오버샘플링 수행 완료 ---")
    print(f"원본 훈련용 클래스 분포: 0={sum(y_train==0)}건, 1={sum(y_train==1)}건")
    print(f"오버샘플링 후 클래스 분포: 0={sum(y_train_res==0)}건, 1={sum(y_train_res==1)}건")
except ImportError:
    # imblearn 패키지 미설치 시 대안 : 지능형 노이즈 가미 간이 복제 함수
    print("imblearn 라이브러리가 없어 간이 노이즈 가미 오버샘플링을 대체 수행합니다.")
    minority_idx = np.where(y_train == 1)[0]
    majority_idx = np.where(y_train == 0)[0]
    multiplier = len(majority_idx) // len(minority_idx) - 1
    
    minority_X = X_train.iloc[minority_idx]
    replicated_X = []
    for _ in range(multiplier):
        # 미세 노이즈 추가 복제
        replicated_X.append(minority_X + np.random.normal(0, 0.01, size=minority_X.shape))
    
    X_train_res = pd.concat([X_train] + replicated_X, axis=0)
    y_train_res = pd.Series(list(y_train) + [1] * (len(minority_X) * multiplier))
    print(f"간이 복제 후 클래스 분포: 0={sum(y_train_res==0)}건, 1={sum(y_train_res==1)}건")

# 전처리 및 샘플링 완료된 훈련용/검증용 데이터를 저장
X_train_res.to_csv('./data/cmp_X_train_res.csv', index=False)
X_test.to_csv('./data/cmp_X_test.csv', index=False)
pd.DataFrame({'failure': y_train_res}).to_csv('./data/cmp_y_train_res.csv', index=False)
pd.DataFrame({'failure': y_test}).to_csv('./data/cmp_y_test.csv', index=False)

print("\nPhase 2 피처 데이터셋 압축 저장 완료!")